# WLS-Var investigation — sims + real-data biorep-split

Builds on `tests/benchmark/bench_linear_weights.py` with a **peptide-level generative model**
(protein × peptide × timepoint) and a **real-data biorep-split** validation. Estimator defs and the
eBayes moderation: `reports/2026-07-24_linear_wls_per_point_var.md`.

`simulate_protein`: each protein has `n_pep` peptides with an intrinsic k-offset **shared across
conditions** (paired design → cancels in Δk). *collapse* = one inverse-variance θ per (biorep, t)
cell (`Var=1/Σ(1/σ²)`, dispersion-biased); *pooled* = every peptide-point (unbiased var, but
pseudoreplicated → **never a default**). Metric: `reject` = Δk rejection rate (Type-I when kA=kB).

In [1]:
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd
from wls_var_sim import run, d0_grid, fit_fdist, simulate_protein, collapse, biorep_split
pd.set_option("display.width", 140); NSIM = 200

## Scenario 2 — collapse vs pooled (`n_pep=4`)

In [2]:
print("TYPE-I  (kA=kB=0.10)"); display(run(0.10, 0.10, 4, NSIM, 1))
print("\nPOWER   (kA=0.10, kB=0.14)"); display(run(0.10, 0.14, 4, NSIM, 3))

TYPE-I  (kA=kB=0.10)


,method,rollup,n,dk_bias,dk_rmse,reject
0,wls_fit,collapse,200,0.000417,0.002945,0.040
1,wls_fit,pooled,200,-0.000191,0.003834,0.005
2,wls_fit_var,collapse,200,0.000551,0.003954,0.090
3,wls_fit_var,pooled,200,0.000529,0.004508,0.075
4,ebayes,collapse,200,0.000533,0.003466,0.070
5,ebayes,pooled,200,0.000298,0.002900,0.005



POWER   (kA=0.10, kB=0.14)


,method,rollup,n,dk_bias,dk_rmse,reject
0,wls_fit,collapse,200,0.000644,0.008417,1.0
1,wls_fit,pooled,200,0.003306,0.008511,1.0
2,wls_fit_var,collapse,200,0.000653,0.009143,1.0
3,wls_fit_var,pooled,200,0.002031,0.009616,1.0
4,ebayes,collapse,200,0.000638,0.008888,1.0
5,ebayes,pooled,200,0.002408,0.007873,1.0


`wls_fit` ~nominal (collapse) / conservative (pooled); raw `wls_fit_var` inflates Type-I under both;
**eBayes fixes it, pooled+eBayes safest** — but pooled pseudoreplicates peptides, so it stays an
*investigation* arm, never a default (rigorous stats want biological replicates, not repeated peptides).

## Scenario 1 — single-peptide proteins (`n_pep=1`)

In [3]:
display(run(0.10, 0.10, 1, NSIM, 2))

,method,rollup,n,dk_bias,dk_rmse,reject
0,wls_fit,collapse,200,0.000038,0.005959,0.025
1,wls_fit,pooled,200,0.000038,0.005959,0.025
2,wls_fit_var,collapse,200,0.000556,0.007771,0.135
3,wls_fit_var,pooled,200,0.000556,0.007771,0.135
4,ebayes,collapse,200,0.000365,0.006525,0.060
5,ebayes,pooled,200,0.000365,0.006525,0.060


Per-point weighting **hurts** (RMSE↑, Type-I↑, no gain); **eBayes shrinks back to `wls_fit`** and
protects the few-peptide cases. The moderated estimator correctly does ≈nothing here.

## d₀ frontier + Smyth `fitFDist` (with limma-robust Winsorization)

Is the data-estimated d₀ optimal, or just sensible? And do heavy-tailed variances need robustifying?

In [4]:
print("eBayes d0 sweep (collapse, TYPE-I at kA=kB=0.10)"); display(d0_grid(0.10, 0.10, 4, NSIM, 1))
# robustness of fitFDist to outlier variances (limma robust=TRUE flavour), boomi collapsed cells:
from wls_var_sim import collapse_cells
c = collapse_cells(pd.read_table("../runs/boomi_ipsc_d2o/riana_fit_fractions.txt", comment="#"))
for trim in (0.0, 0.05, 0.10):
    d0, s0 = fit_fdist(c["var"].to_numpy(), c["df"].to_numpy(), trim=trim)
    print(f"  fitFDist trim {trim:.0%}: d0 = {d0:.2f}")

eBayes d0 sweep (collapse, TYPE-I at kA=kB=0.10)


,d0,reject,dk_rmse
0,0.5,0.085,0.003731
1,1.0,0.080,0.003609
2,2.0,0.070,0.003466
3,4.0,0.055,0.003321
4,8.0,0.050,0.003193


  fitFDist trim 0%: d0 = 1.44
  fitFDist trim 5%: d0 = 1.58
  fitFDist trim 10%: d0 = 1.81


`fitFDist` returns a sensible d₀ but the raw estimate under-shrinks on heavy-tailed proteomics
variances — **Winsorizing (limma `robust=TRUE`) raises d₀ ~25%** (1.44→1.81 on boomi), the safer value.
Production should use the robust estimate (or fall back to fixed d₀=2).

## Real-data validation — biorep-split stability

No ground truth, so use the **biological replicates**: fit per-protein k separately per biorep, and
measure `|log(k_b1/k_b2)|` within each (protein, condition) under `wls` vs `wls-var` (eBayes, robust
d₀). **Lower = more consistent across biological replicates = better** — a fairer test than the
equal-weight per-peptide median (which is itself just an estimator, not truth; that lve test was
inconclusive).

In [5]:
for run_dir in ["boomi_ipsc_d2o", "boomi_ipsc_o18", "timeseries_lauren5_7_ipsc_mesoderm_o18"]:
    tbl, d0 = biorep_split(f"../runs/{run_dir}", d0_trim=0.10)
    if tbl is None: print(f"{run_dir}: <2 bioreps"); continue
    imp = (tbl.loc['wls','multi_pep'] - tbl.loc['wls-var','multi_pep']) / tbl.loc['wls','multi_pep'] * 100
    print(f"### {run_dir}  (robust d0={d0:.2f})   multi-peptide biorep consistency: {imp:+.1f}%")
    display(tbl.round(4))

### boomi_ipsc_d2o  (robust d0=1.81)   multi-peptide biorep consistency: +10.7%


,all,multi_pep,single_pep,n
wls,0.1001,0.0768,0.1866,2747.0
wls-var,0.0926,0.0686,0.1881,2747.0


### boomi_ipsc_o18  (robust d0=1.78)   multi-peptide biorep consistency: +10.4%


,all,multi_pep,single_pep,n
wls,0.0890,0.0599,0.1766,2572.0
wls-var,0.0796,0.0537,0.1777,2572.0


### timeseries_lauren5_7_ipsc_mesoderm_o18  (robust d0=1.61)   multi-peptide biorep consistency: +9.9%


,all,multi_pep,single_pep,n
wls,0.1743,0.1221,0.3056,5749.0
wls-var,0.1561,0.1100,0.3133,5749.0


**Replicated finding:** `wls-var` improves multi-peptide biorep consistency **~10%** across
D₂O + ¹⁸O, iPSC + mesoderm, single/two-condition — with ≈no cost on single-peptide. `juber_ac16_d2o`
is 1-biorep (can't split); `timeseries_dia` (3 biorep × 3 tp) has no run artifact yet — both TODO.
This is the real-world green light for `wls-var` (opt-in, robust d₀, eBayes protecting single-peptide).